In [3]:
import os
from dotenv import load_dotenv
from langgraph.graph import StateGraph,END
from langchain_groq import ChatGroq
from langchain_google_genai import ChatGoogleGenerativeAI
from qdrant_client import QdrantClient
from sentence_transformers import SentenceTransformer, CrossEncoder
from neo4j import GraphDatabase
from typing import TypedDict, List
from langchain_core.output_parsers import StrOutputParser
import warnings

warnings.filterwarnings("ignore")
load_dotenv()


True

In [4]:
qdrant_client = QdrantClient(
    url= os.getenv("QDRANT_URI"),
    api_key=os.getenv("QDRANT_API_KEY")
)

print(qdrant_client.get_collections())

NEO4J_URI = os.getenv("NEO4J_URI")
NEO4J_USERNAME =  os.getenv("NEO4J_USERNAME")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")

driver = GraphDatabase.driver(
    NEO4J_URI,
    auth=(NEO4J_USERNAME, NEO4J_PASSWORD)
)
try:
    driver.verify_connectivity()
    print("Connection established successfully.")
except Exception as e:
    print(f"Connection failed: {e}")

collections=[CollectionDescription(name='Adaptive_RAG')]
Connection established successfully.


In [5]:
gemini_api = os.getenv("GOOGLE_API_KEY")

final_generation = ChatGoogleGenerativeAI(
    model = "gemini-2.5-flash",
    api_key = gemini_api,
    temperature = 0.0
)

In [6]:
retrieval_and_refinement_model = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,
    api_key=os.getenv("GROQ_API_KEY")
)

judge_and_router_model =ChatGroq(
    model = "openai/gpt-oss-20b",
    temperature=0,
    api_key=os.getenv("GROQ_API_KEY")
)



In [7]:
embeddings_model = SentenceTransformer("NeuML/pubmedbert-base-embeddings")
reranker_model = CrossEncoder('cross-encoder/ms-marco-MiniLM-L6-v2')

In [8]:
class AgentNode(TypedDict):
    original_query: str
    refined_query:str
    router_decision:str
    vector_docs:List[str]
    graph_docs:List[str]
    hybrid_docs:List[str]
    reranker_docs:List[str]
    judge_decision:str
    answer:str
    retry_count:int



In [9]:
def agent_router(state: AgentNode):
    query =state['original_query']

    prompt = f"""
    You are a retrieval strategy selector for medical queries about Endoscopic Pituitary Surgery.

    Choose EXACTLY one of these three words based on the query:
    - vector → semantic similarity search needed
    - graph  → entity relationships or structured reasoning needed
    - hybrid → both semantic and relational reasoning needed

    Query: {query}

    Rules:
    - Reply with ONLY one word: vector, graph, or hybrid
    - Do NOT explain, do NOT output anything else
    - Do NOT output any medical terms
    """

    decision = judge_and_router_model.invoke(prompt).content.strip().lower()

    return {"router_decision" : decision}


In [10]:
def vector_retriever(state: AgentNode):

    query = state['original_query']
    top_k = 5
    score_threshold = 0.5
    print("*" * 5, "Querying Qdrant Collection", "*" * 5)

    try:
        query_embeddings = embeddings_model.encode([query],show_progress_bar=True)[0]

        results = qdrant_client.query_points(
            collection_name = "Adaptive_RAG",
            query = query_embeddings.tolist(),
            limit = top_k
        )

        retrieved_docs = []

        for i, result in enumerate(results):
            retrieved_docs.append({
                "id": result.id,
                "content": result.payload.get("text", ""),
                "metadata": result.payload,
                "similarity_score": result.score,
                "rank": i + 1
            })
        print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
        return {
            "vector_docs": retrieved_docs
        }

    except Exception as e:
        print(f"Error during Qdrant retrieval: {e}")
        return {
            "vector_docs": []
        }


In [11]:
def graph_retrieval(state: AgentNode):
    query = state['original_query']
    top_k = 5
    print(f"Graph Retrieval Query: {query}")
    print("*" * 5, "Querying Neo4j Graph", "*" * 5)

    try:

        query_embedding = embeddings_model.encode([query], show_progress_bar=True)[0]
        with driver.session() as session:
            result = session.run(
                """
                MATCH (c:Chunk)
                WITH c, gds.similarity.cosine(c.embedding, $query_embedding) AS score
                ORDER BY score DESC
                LIMIT $top_k
                RETURN c.id AS chunk_id,
                       c.text AS text,
                       score
                """,
                query_embedding=query_embedding.tolist(),
                top_k=top_k
            )
            top_chunks = [r.data() for r in result]

        if not top_chunks:
            print("No graph chunks found")
            return {"graph_docs": []}

        chunk_ids = [r["chunk_id"] for r in top_chunks]
        with driver.session() as session:
            result = session.run(
                """
                MATCH (c:Chunk)-[:MENTIONS]->(e)
                WHERE c.id IN $chunk_ids
                MATCH (e)-[r]-(neighbor)
                RETURN c.id AS chunk_id,
                       c.text AS chunk_text,
                       e.name AS entity,
                       type(r) AS relation,
                       neighbor.name AS neighbor
                """,
                chunk_ids=chunk_ids
            )

            expanded_relations = [r.data() for r in result]
        graph_docs = []

        for chunk in top_chunks:
            graph_docs.append({
                "chunk_id": chunk["chunk_id"],
                "content": chunk["text"],
                "similarity_score": chunk["score"]
            })

        for rel in expanded_relations:
            graph_docs.append({
                "chunk_id": rel["chunk_id"],
                "entity": rel["entity"],
                "relation": rel["relation"],
                "neighbor": rel["neighbor"],
                "content": f"{rel['entity']} --{rel['relation']}--> {rel['neighbor']}"
            })

        print(f"Retrieved {len(graph_docs)} graph-enriched documents")
        return {
            "graph_docs": graph_docs
        }

    except Exception as e:
        print(f"Error during graph retrieval: {e}")
        return {
            "graph_docs": []
        }



In [12]:
def hybrid_retrieve(state):
    query = state["original_query"]

    print("***** SIMPLE HYBRID RETRIEVAL *****")

    vector_docs = vector_retriever(state)

    graph_docs = graph_retrieval(state)
    final_context = vector_docs['vector_docs'] + graph_docs['graph_docs']

    print(f"Hybrid retrieved {len(final_context)} documents")

    return {
        "hybrid_docs": final_context
    }

In [13]:
def rerank_node(state):
    query = state["original_query"]

    docs = (
        state.get("hybrid_docs")
        or state.get("vector_docs")
        or state.get("graph_docs")
        or []
    )

    if not docs:
        print("No docs to rerank")
        return {"reranker_docs": []}

    meaningful_docs = [
        d for d in docs
        if len(d.get("content", "")) > 80   # relation triples are short, skip them
    ]

    print(f"***** RERANKING {len(meaningful_docs)} meaningful docs *****")

    if not meaningful_docs:
        print("⚠️ No meaningful docs after filtering!")
        return {"reranker_docs": []}

    pairs = [(query, d["content"]) for d in meaningful_docs]
    scores = reranker_model.predict(pairs)

    scored_docs = sorted(zip(meaningful_docs, scores), key=lambda x: x[1], reverse=True)

    for doc, score in scored_docs:
        doc["rerank_score"] = float(score)

    top_docs = [doc for doc, _ in scored_docs][:5]

    print(f"Top reranked scores: {[round(d['rerank_score'], 3) for d in top_docs]}")
    return {"reranker_docs": top_docs}

In [14]:
def judge_node(state):
    query = state["original_query"]
    answer = state["answer"]
    docs = state.get("reranker_docs", [])

    context = "\n\n".join([d["content"] for d in docs])

    judge_prompt = f"""You are a medical answer evaluator.

Question: {query}

Context Retrieved:
{context}

Answer Given:
{answer}

Evaluate: Does the answer address the question using information from the context?

Rules:
- If the answer provides relevant medical information about the topic, say YES
- Only say NO if the answer is completely wrong or says "INSUFFICIENT CONTEXT"
- Do NOT be overly strict

Reply with ONLY: YES or NO
"""

    verdict = "NO"  # ✅ default value BEFORE the try block

    try:
        response = judge_and_router_model.invoke(judge_prompt).content.strip().upper()

        if response in {"YES", "NO"}:
            verdict = response
        else:
            print(f"⚠️ Judge returned unexpected: '{response}' → defaulting to NO")

    except Exception as e:
        print(f"⚠️ Judge LLM call failed: {e} → defaulting to NO")

    print(f"Judge verdict: {verdict}")
    return {"judge_decision": verdict}

In [15]:
def refine_query_node(state: AgentNode):
    query = state["original_query"]
    answer = state.get("answer", "")

    refine_prompt = f"""
    The previous retrieval did not return a good answer.

    Original Question: {query}
    Previous Answer: {answer}

    Your knowledge graph contains these entity types:
    - Biological_structure (anatomical parts, organs, tissues)
    - Disease_disorder (conditions, syndromes)
    - Medication (drugs, treatments)
    - Diagnostic_procedure (tests, scans, imaging)
    - Therapeutic_procedure (surgeries, interventions)
    - Sign_symptom (clinical findings)

    Rules for rewriting:
    1. Keep the question SHORT and SIMPLE (max 1 sentence)
    2. Focus on ONE entity or ONE concept only
    3. Use the exact medical term from the original question
    4. Do NOT add extra sub-questions or expand scope
    5. Pick the most relevant entity type and center the question on it

    Example:
    Bad:  "What is the Diaphragma Sellae, its location, role in pituitary and surrounding structures?"
    Good: "What is the Biological_structure Diaphragma Sellae?"

    Original Question: {query}
    Rewritten Question:
    """

    new_query = retrieval_and_refinement_model.invoke(refine_prompt).content.strip()
    print(f"Refined Query: {new_query}")

    return {
        "refined_query": new_query,
        "original_query": new_query,
        "retry_count": state.get("retry_count", 0) + 1
    }

In [21]:
def answer_node(state: AgentNode):
    query = state.get("refined_query") or state["original_query"]
    docs = state.get("reranker_docs", [])

    context = "\n\n".join([d["content"] for d in docs])

    prompt = f"""You are a medical assistant specializing in Endoscopic Pituitary Surgery.
Answer the question using the context below. Be detailed and specific.

Context:
{context}

Question: {query}

Answer:"""

    response = retrieval_and_refinement_model.invoke(prompt)
    new_answer = response.content

    previous_best = state.get("answer", "")
    best_answer = (
        new_answer
        if "INSUFFICIENT CONTEXT" not in new_answer
        else previous_best or new_answer  # fallback to previous if new is bad
    )

    return {"answer": best_answer}

In [22]:
def judge_router(state: AgentNode) -> str:
    max_retries = 3

    if state["judge_decision"] == "YES":
        return "final"
    elif state.get("retry_count", 0) >= max_retries:
        print(f"⚠️ Max retries reached. Returning best available answer.")
        print(f"Final Answer: {state.get('answer', 'None')[:200]}")
        return "final"
    else:
        return "refine"

In [23]:
def route_by_decision(state: AgentNode) -> str:
    return state["router_decision"]

builder = StateGraph(AgentNode)

builder.add_node("router", agent_router)
builder.add_node("vector", vector_retriever)
builder.add_node("graph", graph_retrieval)
builder.add_node("hybrid", hybrid_retrieve)
builder.add_node("rerank", rerank_node)
builder.add_node("answer", answer_node)
builder.add_node("judge", judge_node)
builder.add_node("refine", refine_query_node)

builder.set_entry_point("router")

builder.add_conditional_edges(
    "router",
    route_by_decision,
    {
        "vector": "vector",
        "graph": "graph",
        "hybrid": "hybrid",
    },
)

builder.add_edge("vector", "rerank")
builder.add_edge("graph", "rerank")
builder.add_edge("hybrid", "rerank")
builder.add_edge("rerank", "answer")
builder.add_edge("answer", "judge")

builder.add_conditional_edges(
    "judge",
    judge_router,
    {
        "final": END,
        "refine": "refine",
    },
)
builder.add_edge("refine", "router")

graph = builder.compile()

In [24]:
from IPython.display import Image, display
from langchain_core.runnables.graph import MermaidDrawMethod

# Print the mermaid diagram code
print(graph.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	router(router)
	vector(vector)
	graph(graph)
	hybrid(hybrid)
	rerank(rerank)
	answer(answer)
	judge(judge)
	refine(refine)
	__end__([<p>__end__</p>]):::last
	__start__ --> router;
	answer --> judge;
	graph --> rerank;
	hybrid --> rerank;
	judge -. &nbsp;final&nbsp; .-> __end__;
	judge -.-> refine;
	refine --> router;
	rerank --> answer;
	router -.-> graph;
	router -.-> hybrid;
	router -.-> vector;
	vector --> rerank;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



In [25]:
initial_state = {
    "original_query": "what is Diaphragma Sellae and explain it",
    "iteration_count": 0
}

results = graph.invoke(initial_state)

print("\n===== FINAL OUTPUT =====")
print(results["answer"])

***** Querying Qdrant Collection *****


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Error during Qdrant retrieval: 'tuple' object has no attribute 'id'
No docs to rerank
Judge verdict: YES

===== FINAL OUTPUT =====
The Diaphragma Sellae is a crucial anatomical structure in the context of Endoscopic Pituitary Surgery. It is a thin, fibrous membrane that forms the roof of the sella turcica, a bony cavity located at the base of the skull. The sella turcica is a saddle-shaped depression that houses the pituitary gland, a vital endocrine gland responsible for regulating various bodily functions.

The Diaphragma Sellae is a dural fold that separates the pituitary gland from the suprasellar cistern, a fluid-filled space that contains cerebrospinal fluid (CSF). This membrane is composed of two layers of dura mater, a tough, fibrous tissue that surrounds the brain and spinal cord. The Diaphragma Sellae has a central aperture, which allows the pituitary stalk to pass through, connecting the pituitary gland to the hypothalamus.

In Endoscopic Pituitary Surgery, the Diaphragma Se